In [ ]:
# ============================================
# AI FOOD ORDERING CHATBOT
# Built using Anthropic's Claude API
# ============================================

# Install required libraries
# anthropic  = library to interact with Claude API
# python-dotenv = library to securely load API keys
%pip install anthropic python-dotenv

# Import API key from Google Colab secrets
# (Add your ANTHROPIC_API_KEY in Colab's secrets section)
from google.colab import userdata
from anthropic import Anthropic

# Create a connection to Claude
# Think of "client" as picking up the phone to call Claude
client = Anthropic(api_key=userdata.get('ANTHROPIC_API_KEY'))

# Define which Claude model to use
# claude-sonnet-4-0 = fast, smart, and cost-effective
model = "claude-sonnet-4-0"


# ============================================
# CHAT FUNCTION
# Sends messages to Claude and streams the response
# back word by word — just like ChatGPT!
# ============================================

def chat(messages, system=None, temperature=1.0, stop_sequences=None):
    """
    Parameters:
    - messages      : full conversation history (list of dicts)
    - system        : instructions that define Claude's role (optional)
    - temperature   : creativity level between 0.0 and 1.0 (default 1.0)
    - stop_sequences: list of strings where Claude should stop generating (optional)

    Returns:
    - Claude's complete response as a string
    """

    # Build the parameters dictionary for the API call
    params = {
        "model": model,
        "max_tokens": 1000,       # maximum length of Claude's response
        "messages": messages,      # full conversation history
        "temperature": temperature # controls creativity vs consistency
    }

    # Only add system prompt if one was provided
    # The API does not accept system=None, so we check first
    if system:
        params["system"] = system

    # Only add stop sequences if they were provided
    if stop_sequences:
        params["stop_sequences"] = stop_sequences

    # Stream Claude's response word by word
    # "with" ensures the stream is automatically closed after use
    with client.messages.stream(**params) as stream:

        # Empty string to collect all chunks as they arrive
        full_answer = ""

        # Each iteration receives a small piece of the response
        for text in stream.text_stream:
            print(text, end="", flush=True)
            # end=""     = do not move to a new line after each chunk
            # flush=True = display immediately without buffering

            # Add each chunk to build the complete response
            # This is needed to save the full reply to conversation history
            full_answer += text

        # Move to a new line once the full response has been received
        print()

    # Return the complete response for saving to history
    return full_answer


# ============================================
# SYSTEM PROMPT
# This is Claude's instruction manual
# It defines exactly how the chatbot should behave
# Think of it as training a new employee
# ============================================

system_prompt = """
You are a friendly and professional food ordering assistant
for a restaurant called "Harry's Kitchen".

YOUR MENU:
- Margherita Pizza    : $8.99  (classic tomato and cheese)
- Pepperoni Pizza     : $10.99 (pepperoni with extra cheese)
- BBQ Chicken Pizza   : $11.99 (grilled chicken, BBQ sauce)
- Veggie Pizza        : $9.99  (bell peppers, mushrooms, olives)
- Garlic Bread        : $3.99  (toasted with garlic butter)
- Caesar Salad        : $6.99  (romaine lettuce, croutons, parmesan)
- Coke / Pepsi        : $1.99
- Water               : $0.99

YOUR RESPONSIBILITIES:
1. Greet every customer warmly when they arrive
2. Help customers choose items from the menu only
3. Remember the customer's full order throughout the conversation
4. When the customer is done, provide a complete itemized bill
5. Calculate the total price accurately
6. Thank the customer at the end of every order

IMPORTANT RULES:
- Do NOT discuss anything unrelated to food ordering
- Do NOT suggest items that are not listed on the menu
- Always confirm the full order before presenting the final bill
- Be patient, friendly, and helpful at all times
"""


# ============================================
# CONVERSATION HISTORY
# This list acts as Claude's memory
# Every message — from both the user and Claude —
# is stored here and sent with every API call
# Without this, Claude would forget the conversation instantly
#
# Each message follows this structure:
# {"role": "user",      "content": "..."}  = customer's message
# {"role": "assistant", "content": "..."}  = Claude's response
# ============================================

conversation_history = []


# ============================================
# MAIN CHATBOT LOOP
# The restaurant stays open until the customer says "bye"
# Each iteration: take input, send to Claude, print response
# ============================================

print("Welcome to Harry's Kitchen!")
print("Type 'bye' to exit\n")

while True:

    # Take the customer's input
    user_input = input("You: ")

    # If the customer says "bye", end the conversation
    if user_input.lower() == "bye":
        print("Harry's Kitchen: Thank you for visiting! Goodbye!")
        break

    # Save the customer's message to conversation history
    # role = "user" because the customer sent this message
    conversation_history.append({
        "role": "user",
        "content": user_input
    })

    # Send the full conversation history to Claude and get a response
    # Passing history every time is what gives Claude its "memory"
    print("Harry's Kitchen: ", end="")
    answer = chat(
        conversation_history,    # full conversation so far
        system=system_prompt,    # restaurant assistant instructions
        temperature=0.7          # balanced between creative and consistent
    )

    # Save Claude's response to conversation history
    # role = "assistant" because Claude sent this message
    # This is what allows Claude to remember what it said earlier
    conversation_history.append({
        "role": "assistant",
        "content": answer
    })